[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-creditscore.ipynb)

# Full Project: Credit Score Classification

*AIBits Academy · Machine Learning End To End · Full Project*

100,000 genuinely messy bank-customer records — garbage age values, string-encoded numbers, and inconsistent formatting — classified into Good/Standard/Poor credit tiers.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['credit_score_classification.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the raw credit-history table** (100,000 rows). Several numeric columns arrive as text with data-entry artifacts, exactly as the lesson describes.

In [ ]:
df = pd.read_csv('credit_score_classification.csv', low_memory=False)
print(df.shape)

> **Business Problem**
>
> A financial company wants to classify customers into credit-score brackets (Poor / Standard / Good) automatically, instead of relying on a manual credit-bureau lookup for every applicant, to speed up loan and credit-card approval decisions.

> **Dataset**
>
> **100,000 rows, 28 columns** of customer financial history (income, bank accounts, credit cards, loan delays, credit mix, EMI, monthly balance). [Dataset source →](https://statso.io/credit-score-classification-case-study/)

## Step 1 — This Data Is Genuinely Messy

Unlike most datasets used elsewhere in this course, several numeric columns here are stored as text with data-entry artifacts — a direct, real illustration of the Data Preprocessing page's cleaning techniques:

In [ ]:
print(df['Age'].unique()[:10])

Ages of `-500` and trailing underscores like `28_` are data-entry corruption, not real values. The cleaning pass: strip stray underscores and coerce to numeric, then treat anything outside a plausible 14–100 range as missing rather than a real age:

In [ ]:
def clean_numeric(col):
    return pd.to_numeric(col.astype(str).str.replace('_',''), errors='coerce')

for col in ['Age','Annual_Income','Outstanding_Debt','Amount_invested_monthly','Monthly_Balance']:
    df[col] = clean_numeric(df[col])

df.loc[(df['Age']<14) | (df['Age']>100), 'Age'] = np.nan
print(f"Age range after cleaning: {df['Age'].min()} to {df['Age'].max()}")

The `Credit_History_Age` column stores values like `"22 Years and 1 Months"` as free text — parsed into a single numeric "total months" feature via regex before it can be used by any model.

## Step 2 — Random Forest Classification

17 numeric features (income, EMI, delay counts, utilization ratio, parsed credit history length, ...) plus 4 encoded categoricals (Credit_Mix, Payment_of_Min_Amount, Payment_Behaviour, Occupation), median-imputed and label-encoded, fed into a Random Forest across 3 classes:

Finish preparing the model matrix as the text describes: parse `Credit_History_Age` ("22 Years and 1 Months") into months, clean the remaining text-typed numbers, median-impute, and label-encode the four categoricals.

In [ ]:
import re
from sklearn.preprocessing import LabelEncoder

def history_to_months(s):
    m = re.match(r"(\d+) Years and (\d+) Months", str(s))
    return int(m.group(1)) * 12 + int(m.group(2)) if m else np.nan

df['Credit_History_Age'] = df['Credit_History_Age'].apply(history_to_months)
for col in ['Num_of_Loan', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit']:
    df[col] = clean_numeric(df[col])

num_features = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
                'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
                'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month',
                'Amount_invested_monthly', 'Monthly_Balance']
cat_features = ['Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour', 'Occupation']

X = df[num_features + cat_features].copy()
X[num_features] = X[num_features].fillna(X[num_features].median())
for c in cat_features:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))
y = df['Credit_Score']
print(X.shape, y.value_counts().to_dict())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Macro F1: {f1_score(y_test, preds, average='macro'):.4f}")

The model performs best on the majority "Standard" class (F1 0.80) and weakest on "Good" (F1 0.71, also the smallest class at 17,828 of 100,000 rows) — the familiar class-imbalance pattern from the Handling Imbalanced Data page, here across 3 classes rather than 2.

## Visualizing the Per-Class Breakdown

All three metrics tell the same story in the same direction — "Good," the smallest class, is weakest on every one of them.

## Step 3 — What Actually Drives Credit Score?

| Feature | Importance |
|---|---|
| Outstanding_Debt | 0.1205 |
| Interest_Rate | 0.0939 |
| Credit_Mix | 0.0751 |
| Delay_from_due_date | 0.0687 |
| Changed_Credit_Limit | 0.0591 |
| Credit_History_Age (months) | 0.0542 |

Outstanding debt and interest rate dominate — consistent with real credit-scoring practice, where existing debt burden and the rate a lender already charges (itself a proxy for prior perceived risk) are the strongest signals.

## Key Business Takeaways

- Real financial data arrives with data-entry corruption (`-500` as an age, stray underscores) that must be caught before modeling, not assumed away.
- 77.5% accuracy across 3 imbalanced classes is a solid result, but the weakest class (Good, F1 0.71) is also the smallest — exactly where the Handling Imbalanced Data page's techniques (class weighting, SMOTE) would be the natural next step.
- Outstanding debt and interest rate — not income alone — are the strongest predictors, a reminder that raw income is a weaker signal than how a customer manages existing credit.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Clean an age column

Write `clean_age(s)`: strip underscores, convert to numbers (`errors="coerce"`), and turn anything outside 14-100 into `NaN`. Test it on the sample below.

In [ ]:
import pandas as pd, numpy as np
def clean_age(s):
    pass   # TODO
sample = pd.Series(["23", "-500", "28_", "34", "1200"])


In [ ]:
try:
    out = clean_age(sample)
    check("23 and 34 stay", out.iloc[0] == 23 and out.iloc[3] == 34)
    check("28_ becomes 28", out.iloc[2] == 28)
    check("impossible ages become NaN", pd.isna(out.iloc[1]) and pd.isna(out.iloc[4]))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd, numpy as np
def clean_age(s):
    a = pd.to_numeric(s.astype(str).str.replace("_", ""), errors="coerce")
    return a.where((a >= 14) & (a <= 100))
sample = pd.Series(["23", "-500", "28_", "34", "1200"])

```

</details>

### Exercise 2 · Medium · Class shares

Store the share of each `Credit_Score` class in `class_share` (a Series). Which class is smallest?

In [ ]:
class_share = None   # TODO


In [ ]:
try:
    check("Standard about 53%", abs(class_share["Standard"] - 0.53174) < 1e-9)
    check("Good is smallest", class_share.idxmin() == "Good")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
class_share = y.value_counts(normalize=True)

```

</details>

### Exercise 3 · Stretch · What drives the score?

Store in `top3` the names of the three most important features of the lesson's fitted forest `rf` (most important first). The lesson reports outstanding debt as number one.

In [ ]:
top3 = None   # TODO (feature names are X.columns)


In [ ]:
try:
    check("three names", len(top3) == 3)
    check("outstanding debt is in the top 3", "Outstanding_Debt" in top3)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
top3 = list(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).index[:3])

```

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Credit Score Classification**.*